# Route B — Static Word Embeddings + Classical Classifiers

Route B sits between Route A (sparse bag-of-words) and Route C (contextual transformer) on the
representation ladder. Each document becomes a **single dense vector** by averaging pretrained
word vectors, then a classical classifier learns the sentiment boundary. The embeddings are
neural-derived, but **no network runs at inference** — which is exactly the narrative step the
project is built around.

We compare three pretrained embedding sets (**GloVe**, **word2vec**, **fastText**) and two
pooling schemes (plain mean vs **tf-idf-weighted** mean), all on the shared split. Because the
final feature space is small and dense, a non-linear **RBF SVC** is cheap here and included.

The fitted embedding lookup is **baked into the `.model`**, so deployment needs only
scikit-learn/numpy — `gensim` is a *training-time* dependency only.

### 1/ Imports

In [2]:
import os, re, pickle, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED      = 42
MODEL_OUT = "route_b.model"
N_SPLITS  = 5
TEXT_COL  = "Text"          # raw text column (embeddings prefer un-cleaned text)
np.random.seed(SEED)
_TOK = re.compile(r"[a-z]+")

### 2/ Load the shared split

In [3]:
data = pd.read_csv("data_split.csv", index_col=0)
data[TEXT_COL] = data[TEXT_COL].fillna("").astype(str)
data["sentiment"] = data["sentiment"].astype(int)

train_df = data[data["split"] == "train"].reset_index(drop=True)
val_df   = data[data["split"] == "val"].reset_index(drop=True)
test_df  = data[data["split"] == "test"].reset_index(drop=True)
y_train, y_val, y_test = (d["sentiment"].to_numpy() for d in (train_df, val_df, test_df))
print(f"train {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,}")

train 178,557 | val 38,263 | test 38,263


### 3/ Load pretrained embeddings (training time only)

`gensim.downloader` fetches and caches the vectors. GloVe-100 is the smallest and a sensible
default; word2vec-google-news (300d) and fastText (300d, sub-word, handles OOV better for
tweets) are heavier downloads. Toggle `EMBEDDINGS` to control which sets enter the leaderboard.

In [4]:
import gensim.downloader as api

EMBEDDINGS = {
    #"glove-100":  "glove-wiki-gigaword-100",
    "glove-twitter-200":  "glove-twitter-200",
    "w2v-300":            "word2vec-google-news-300",
    "fasttext-300":       "fasttext-wiki-news-subwords-300",
}

kv = {}
for short, name in EMBEDDINGS.items():
    print(f"loading {name} ...")
    kv[short] = api.load(name)
    print(f"  {short}: {len(kv[short].index_to_key):,} vectors, dim={kv[short].vector_size}")

loading glove-twitter-200 ...
  glove-twitter-200: 1,193,514 vectors, dim=200
loading word2vec-google-news-300 ...
  w2v-300: 3,000,000 vectors, dim=300
loading fasttext-wiki-news-subwords-300 ...
  fasttext-300: 999,999 vectors, dim=300


### 5/ Leaderboard (embedding × pooling × classifier, stratified CV)

Dense features are standardized before the margin/again-based learners. The pre-computed
document vectors are cached per (embedding, pooling) so the CV loop only re-fits the classifier,
keeping it fast.

In [10]:
import re, warnings
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.exceptions import ConvergenceWarning
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import StratifiedKFold, HalvingGridSearchCV
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif    # chi2 won't work: embeddings have negatives
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

SEED = 42
CV5  = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

NEUTRAL_WEIGHTS = [None, "balanced", {-1: 1.0, 0: 1.5, 1: 1.0}]
RF_WEIGHTS      = [None, "balanced", {-1: 1.0, 0: 1.5, 1: 1.0}]

TEXT_COL = "Text"


# ----------------------------------------------------------------------------- encoder
class EmbeddingVectorizer(BaseEstimator, TransformerMixin):
    """Aggregate pretrained word vectors into one dense vector per document.
    Pickle-safe: stores only plain dicts/strings (no KeyedVectors, no external
    _TOK regex, no lambdas) so it round-trips for the API. fit() is a no-op,
    so it's also immune to the case-manual's fit_transform-at-inference bug.
    agg: 'mean' | 'idf' | 'meanmax'
    """
    def __init__(self, wv=None, dim=0, idf=None, agg="mean", token_pattern=r"[a-z]+"):
        self.wv  = wv  if wv  is not None else {}
        self.dim = dim
        self.idf = idf if idf is not None else {}
        self.agg = agg
        self.token_pattern = token_pattern

    def fit(self, X, y=None):          # vectors + idf baked in at build time
        return self

    @property
    def out_dim(self):
        return self.dim * 2 if self.agg == "meanmax" else self.dim

    def _vec(self, text):
        toks = re.findall(self.token_pattern, str(text).lower())
        vs, ws = [], []
        for t in toks:
            v = self.wv.get(t)
            if v is not None:
                vs.append(v)
                ws.append(self.idf.get(t, 1.0) if self.agg == "idf" else 1.0)
        if not vs:
            return np.zeros(self.out_dim, dtype="float32")
        vs = np.asarray(vs, dtype="float32")
        ws = np.asarray(ws, dtype="float32")
        mean = (vs * ws[:, None]).sum(0) / max(ws.sum(), 1e-9)
        if self.agg == "meanmax":
            return np.concatenate([mean, vs.max(0)]).astype("float32")
        return mean.astype("float32")

    def transform(self, X):
        return np.vstack([self._vec(t) for t in X]).astype("float32")


def make_embedding_vectorizer(keyed_vectors, train_texts, agg="mean",
                              token_pattern=r"[a-z]+"):
    """Restrict vectors to TRAIN vocab; precompute idf if agg='idf'.
    All data-driven parts (vocab, idf) use train texts only -> no leakage."""
    vocab = set()
    for t in train_texts:
        vocab.update(re.findall(token_pattern, str(t).lower()))
    wv = {w: keyed_vectors[w].astype("float32")
          for w in vocab if w in keyed_vectors.key_to_index}
    idf = {}
    if agg == "idf":
        tfv = TfidfVectorizer(token_pattern=token_pattern, min_df=2)
        tfv.fit(train_texts)
        idf = {w: float(tfv.idf_[i]) for w, i in tfv.vocabulary_.items() if w in wv}
    return EmbeddingVectorizer(wv=wv, dim=keyed_vectors.vector_size,
                               idf=idf, agg=agg, token_pattern=token_pattern)


# ----------------------------------------------------------------------------- classifier
def _registry_b(clf_name):
    # estimator, clf-param grid (C/class_weight analogue), needs_scale, k grid.
    # NB: ComplementNB dropped — it needs non-negative features, embeddings have negatives.
    if clf_name == "logreg":
        return (LogisticRegression(max_iter=2000),                 # no n_jobs here
                {"clf__C": [0.1, 0.5, 1.0, 2.0],
                 "clf__class_weight": NEUTRAL_WEIGHTS}, True, ["all"])
    if clf_name == "linearSVC":
        return (LinearSVC(dual="auto", max_iter=5000),
                {"clf__C": [0.1, 0.3, 0.5, 1.0],
                 "clf__class_weight": NEUTRAL_WEIGHTS}, True, ["all"])
    if clf_name == "rf":
        return (RandomForestClassifier(n_estimators=300, random_state=SEED),  # no n_jobs here
                {"clf__max_depth": [None, 20, 40],
                 "clf__class_weight": RF_WEIGHTS}, False, ["all"])
    if clf_name == "knn":
        return (KNeighborsClassifier(),                            # no n_jobs here
                {"clf__n_neighbors": [15, 25, 45]}, True, ["all"])
    raise ValueError(f"unknown classifier {clf_name}")


def tune_cell_b(X, source, agg, clf_name):
    """X is the precomputed embedding matrix for (source, agg). The vectorizer
    is OUT of the search (its fit is a no-op), so we tune only the classifier."""
    est, clf_grid, needs_scale, k_grid = _registry_b(clf_name)

    steps = [("sel", SelectKBest(f_classif))]
    if needs_scale:
        steps.append(("scale", StandardScaler()))   # trees skip it
    steps.append(("clf", est))
    pipe = Pipeline(steps)

    param_grid = {"sel__k": k_grid, **clf_grid}

    gs = HalvingGridSearchCV(
        pipe, param_grid, scoring="f1_macro", cv=CV5,
        factor=3, random_state=SEED, n_jobs=1, verbose=1, refit=True)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConvergenceWarning)
        warnings.simplefilter("ignore", category=FutureWarning)
        gs.fit(X, y_train)

    return {"source": source, "agg": agg, "classifier": clf_name,
            "cv_macroF1": round(gs.best_score_, 4), "best_params": gs.best_params_,
            "estimator": gs.best_estimator_}


def print_result_b(r):
    print(f"{r['source']:18} | {r['agg']:7} | {r['classifier']:10} -> "
          f"F1={r['cv_macroF1']:.4f} | {r['best_params']}")


In [7]:
SOURCES = ["glove-twitter-200", "w2v-300", "fasttext-300"]   # keys in your kv dict
AGGS    = ["mean", "idf"]            # add "meanmax" to also test concat pooling
CLFS    = ["logreg", "linearSVC", "rf", "knn"]

# build each vectorizer once, then transform the train texts once per (source, agg)
train_texts = train_df[TEXT_COL].tolist()
VEC = {(s, a): make_embedding_vectorizer(kv[s], train_texts, agg=a)
       for s in SOURCES for a in AGGS}
EMB = {(s, a): VEC[(s, a)].transform(train_texts)            # dense (n, dim) float32
       for s in SOURCES for a in AGGS}

In [8]:
results_b = []

# glove-twitter-200
print("Running: source=glove-twitter-200 | agg=mean | clf=logreg")
r = tune_cell_b(EMB[("glove-twitter-200", "mean")], "glove-twitter-200", "mean", "logreg")
results_b.append(r); print_result_b(r)

Running: source=glove-twitter-200 | agg=mean | clf=logreg
glove-twitter-200  | mean    | logreg     -> F1=0.6208 | {'clf__C': 1.0, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [11]:
print("Running: source=glove-twitter-200 | agg=mean | clf=linearSVC")
r = tune_cell_b(EMB[("glove-twitter-200", "mean")], "glove-twitter-200", "mean", "linearSVC")
results_b.append(r); print_result_b(r)

Running: source=glove-twitter-200 | agg=mean | clf=linearSVC
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 19839
max_resources_: 178557
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 12
n_resources: 19839
Fitting 5 folds for each of 12 candidates, totalling 60 fits
----------
iter: 1
n_candidates: 4
n_resources: 59517
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 2
n_candidates: 2
n_resources: 178551
Fitting 5 folds for each of 2 candidates, totalling 10 fits
glove-twitter-200  | mean    | linearSVC  -> F1=0.6119 | {'clf__C': 0.3, 'clf__class_weight': {-1: 1.0, 0: 1.5, 1: 1.0}, 'sel__k': 'all'}


In [12]:
print("Running: source=glove-twitter-200 | agg=idf | clf=logreg")
r = tune_cell_b(EMB[("glove-twitter-200", "idf")], "glove-twitter-200", "idf", "logreg")
results_b.append(r); print_result_b(r)

Running: source=glove-twitter-200 | agg=idf | clf=logreg
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 19839
max_resources_: 178557
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 12
n_resources: 19839
Fitting 5 folds for each of 12 candidates, totalling 60 fits
----------
iter: 1
n_candidates: 4
n_resources: 59517
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 2
n_candidates: 2
n_resources: 178551
Fitting 5 folds for each of 2 candidates, totalling 10 fits
glove-twitter-200  | idf     | logreg     -> F1=0.6019 | {'clf__C': 2.0, 'clf__class_weight': {-1: 1.0, 0: 1.5, 1: 1.0}, 'sel__k': 'all'}


In [13]:
print("Running: source=glove-twitter-200 | agg=idf | clf=linearSVC")
r = tune_cell_b(EMB[("glove-twitter-200", "idf")], "glove-twitter-200", "idf", "linearSVC")
results_b.append(r); print_result_b(r)

Running: source=glove-twitter-200 | agg=idf | clf=linearSVC
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 19839
max_resources_: 178557
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 12
n_resources: 19839
Fitting 5 folds for each of 12 candidates, totalling 60 fits
----------
iter: 1
n_candidates: 4
n_resources: 59517
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 2
n_candidates: 2
n_resources: 178551
Fitting 5 folds for each of 2 candidates, totalling 10 fits
glove-twitter-200  | idf     | linearSVC  -> F1=0.5945 | {'clf__C': 0.3, 'clf__class_weight': {-1: 1.0, 0: 1.5, 1: 1.0}, 'sel__k': 'all'}


In [14]:
print("Running: source=w2v-300 | agg=mean | clf=logreg")
r = tune_cell_b(EMB[("w2v-300", "mean")], "w2v-300", "mean", "logreg")
results_b.append(r); print_result_b(r)

Running: source=w2v-300 | agg=mean | clf=logreg
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 19839
max_resources_: 178557
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 12
n_resources: 19839
Fitting 5 folds for each of 12 candidates, totalling 60 fits
----------
iter: 1
n_candidates: 4
n_resources: 59517
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 2
n_candidates: 2
n_resources: 178551
Fitting 5 folds for each of 2 candidates, totalling 10 fits
w2v-300            | mean    | logreg     -> F1=0.6264 | {'clf__C': 2.0, 'clf__class_weight': {-1: 1.0, 0: 1.5, 1: 1.0}, 'sel__k': 'all'}


In [15]:
print("Running: source=w2v-300 | agg=mean | clf=linearSVC")
r = tune_cell_b(EMB[("w2v-300", "mean")], "w2v-300", "mean", "linearSVC")
results_b.append(r); print_result_b(r)

Running: source=w2v-300 | agg=mean | clf=linearSVC
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 19839
max_resources_: 178557
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 12
n_resources: 19839
Fitting 5 folds for each of 12 candidates, totalling 60 fits
----------
iter: 1
n_candidates: 4
n_resources: 59517
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 2
n_candidates: 2
n_resources: 178551
Fitting 5 folds for each of 2 candidates, totalling 10 fits
w2v-300            | mean    | linearSVC  -> F1=0.6166 | {'clf__C': 0.5, 'clf__class_weight': {-1: 1.0, 0: 1.5, 1: 1.0}, 'sel__k': 'all'}


In [16]:
print("Running: source=w2v-300 | agg=idf | clf=logreg")
r = tune_cell_b(EMB[("w2v-300", "idf")], "w2v-300", "idf", "logreg")
results_b.append(r); print_result_b(r)

Running: source=w2v-300 | agg=idf | clf=logreg
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 19839
max_resources_: 178557
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 12
n_resources: 19839
Fitting 5 folds for each of 12 candidates, totalling 60 fits
----------
iter: 1
n_candidates: 4
n_resources: 59517
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 2
n_candidates: 2
n_resources: 178551
Fitting 5 folds for each of 2 candidates, totalling 10 fits
w2v-300            | idf     | logreg     -> F1=0.6057 | {'clf__C': 2.0, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [17]:
print("Running: source=w2v-300 | agg=idf | clf=linearSVC")
r = tune_cell_b(EMB[("w2v-300", "idf")], "w2v-300", "idf", "linearSVC")
results_b.append(r); print_result_b(r)

Running: source=w2v-300 | agg=idf | clf=linearSVC
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 19839
max_resources_: 178557
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 12
n_resources: 19839
Fitting 5 folds for each of 12 candidates, totalling 60 fits
----------
iter: 1
n_candidates: 4
n_resources: 59517
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 2
n_candidates: 2
n_resources: 178551
Fitting 5 folds for each of 2 candidates, totalling 10 fits
w2v-300            | idf     | linearSVC  -> F1=0.5953 | {'clf__C': 0.5, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [18]:
# fasttext-300
print("Running: source=fasttext-300 | agg=mean | clf=logreg")
r = tune_cell_b(EMB[("fasttext-300", "mean")], "fasttext-300", "mean", "logreg")
results_b.append(r); print_result_b(r)

Running: source=fasttext-300 | agg=mean | clf=logreg
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 19839
max_resources_: 178557
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 12
n_resources: 19839
Fitting 5 folds for each of 12 candidates, totalling 60 fits
----------
iter: 1
n_candidates: 4
n_resources: 59517
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 2
n_candidates: 2
n_resources: 178551
Fitting 5 folds for each of 2 candidates, totalling 10 fits
fasttext-300       | mean    | logreg     -> F1=0.6313 | {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [19]:
print("Running: source=fasttext-300 | agg=mean | clf=linearSVC")
r = tune_cell_b(EMB[("fasttext-300", "mean")], "fasttext-300", "mean", "linearSVC")
results_b.append(r); print_result_b(r)

Running: source=fasttext-300 | agg=mean | clf=linearSVC
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 19839
max_resources_: 178557
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 12
n_resources: 19839
Fitting 5 folds for each of 12 candidates, totalling 60 fits
----------
iter: 1
n_candidates: 4
n_resources: 59517
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 2
n_candidates: 2
n_resources: 178551
Fitting 5 folds for each of 2 candidates, totalling 10 fits
fasttext-300       | mean    | linearSVC  -> F1=0.6216 | {'clf__C': 1.0, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [20]:
print("Running: source=fasttext-300 | agg=idf | clf=logreg")
r = tune_cell_b(EMB[("fasttext-300", "idf")], "fasttext-300", "idf", "logreg")
results_b.append(r); print_result_b(r)

Running: source=fasttext-300 | agg=idf | clf=logreg
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 19839
max_resources_: 178557
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 12
n_resources: 19839
Fitting 5 folds for each of 12 candidates, totalling 60 fits
----------
iter: 1
n_candidates: 4
n_resources: 59517
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 2
n_candidates: 2
n_resources: 178551
Fitting 5 folds for each of 2 candidates, totalling 10 fits
fasttext-300       | idf     | logreg     -> F1=0.6156 | {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [21]:
print("Running: source=fasttext-300 | agg=idf | clf=linearSVC")
r = tune_cell_b(EMB[("fasttext-300", "idf")], "fasttext-300", "idf", "linearSVC")
results_b.append(r); print_result_b(r)

Running: source=fasttext-300 | agg=idf | clf=linearSVC
n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 19839
max_resources_: 178557
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 12
n_resources: 19839
Fitting 5 folds for each of 12 candidates, totalling 60 fits
----------
iter: 1
n_candidates: 4
n_resources: 59517
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 2
n_candidates: 2
n_resources: 178551
Fitting 5 folds for each of 2 candidates, totalling 10 fits
fasttext-300       | idf     | linearSVC  -> F1=0.6064 | {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [22]:
leaderboard = (
    pd.DataFrame(results_b)
    .drop(columns=["estimator"], errors="ignore")
    .sort_values("cv_macroF1", ascending=False)
    .reset_index(drop=True))

print(leaderboard.to_string(index=False))

best_row = leaderboard.iloc[0]
print("\nWinner:", best_row.to_dict())

           source  agg classifier  cv_macroF1                                                                      best_params
     fasttext-300 mean     logreg      0.6313                {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}
          w2v-300 mean     logreg      0.6264 {'clf__C': 2.0, 'clf__class_weight': {-1: 1.0, 0: 1.5, 1: 1.0}, 'sel__k': 'all'}
     fasttext-300 mean  linearSVC      0.6216                {'clf__C': 1.0, 'clf__class_weight': 'balanced', 'sel__k': 'all'}
glove-twitter-200 mean     logreg      0.6208                {'clf__C': 1.0, 'clf__class_weight': 'balanced', 'sel__k': 'all'}
          w2v-300 mean  linearSVC      0.6166 {'clf__C': 0.5, 'clf__class_weight': {-1: 1.0, 0: 1.5, 1: 1.0}, 'sel__k': 'all'}
     fasttext-300  idf     logreg      0.6156                {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}
glove-twitter-200 mean  linearSVC      0.6119 {'clf__C': 0.3, 'clf__class_weight': {-1: 1.0, 0: 1.5, 1: 1.0}, '

### 6/ Final model, test metrics, per-domain breakdown

In [24]:
best_emb    = best_row["source"]
best_agg    = best_row["agg"]
best_clf    = best_row["classifier"]
best_params = best_row["best_params"]      # e.g. {'clf__C':0.1,'clf__class_weight':'balanced','sel__k':'all'}

# 1) Vectorizer: vocab/idf fit on TRAIN+VAL only (test stays unseen)
X_trval = pd.concat([train_df[TEXT_COL], val_df[TEXT_COL]], ignore_index=True)
y_trval = np.concatenate([y_train, y_val])

mev_final = make_embedding_vectorizer(kv[best_emb], X_trval.tolist(), agg=best_agg)
Xtr = mev_final.transform(X_trval)
Xte = mev_final.transform(test_df[TEXT_COL])

# 2) Rebuild the SAME pipeline used in tuning, then apply the winning params
est, _, needs_scale, _ = _registry_b(best_clf)
steps = [("sel", SelectKBest(f_classif))]
if needs_scale:
    steps.append(("scale", StandardScaler()))   # trees skip it
steps.append(("clf", clone(est)))
pipe = Pipeline(steps).set_params(**best_params)

# 3) Fit on TRAIN+VAL, predict on TEST
pipe.fit(Xtr, y_trval)
y_pred = pipe.predict(Xte)

print(f"Winning config: {best_emb} | {best_agg} | {best_clf} | {best_params}")
print(f"TEST macro-F1: {f1_score(y_test, y_pred, average='macro'):.4f}\n")
print(classification_report(y_test, y_pred, labels=[-1, 0, 1],
      target_names=["Negative", "Neutral", "Positive"]))
print("Confusion matrix [-1,0,1]:")
print(confusion_matrix(y_test, y_pred, labels=[-1, 0, 1]))
print("\nPer-domain macro-F1:")
for dom in ["Review", "Tweet"]:
    m = (test_df["Type"] == dom).to_numpy()
    if m.any():
        print(f"  {dom:>7}: {f1_score(y_test[m], y_pred[m], average='macro'):.4f} (n={m.sum():,})")

Winning config: fasttext-300 | mean | logreg | {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}
TEST macro-F1: 0.6322

              precision    recall  f1-score   support

    Negative       0.69      0.70      0.70     13468
     Neutral       0.45      0.52      0.48      9528
    Positive       0.76      0.68      0.72     15267

    accuracy                           0.65     38263
   macro avg       0.63      0.63      0.63     38263
weighted avg       0.66      0.65      0.65     38263

Confusion matrix [-1,0,1]:
[[ 9483  2709  1276]
 [ 2559  4912  2057]
 [ 1620  3269 10378]]

Per-domain macro-F1:
   Review: 0.6146 (n=30,750)
    Tweet: 0.5096 (n=7,513)
